# Crowd counting with MCNN — Kaggle training

Same pipeline as the Colab notebook, adapted to Kaggle:

- The dataset is **attached as an input**, not downloaded — click
  *Add Input* and search for **ShanghaiTech** (e.g. `tthien/shanghaitech`).
- `/kaggle/input` is read-only, so processed data, checkpoints and
  outputs all go to `/kaggle/working`.
- Turn on the GPU under *Settings -> Accelerator*.

`/kaggle/working` persists in the notebook's output, so download
`best.pth` from there when the run finishes.

## 1. Check the GPU and find the dataset

In [ ]:
!nvidia-smi

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

In [ ]:
!find /kaggle/input -maxdepth 4 -iname 'part_*' -type d | head

## 2. Get the code

Either clone the repo or attach it as a Kaggle dataset / utility script.

In [ ]:
!git clone -q https://github.com/jugalkkt/crowd-density-maps.git /kaggle/working/crowd-counting-mcnn
!pip install -q -r /kaggle/working/crowd-counting-mcnn/requirements.txt

In [ ]:
%cd /kaggle/working/crowd-counting-mcnn
!ls

## 3. Paths

Edit `RAW_DIR` to match the `find` output above.

In [ ]:
RAW_DIR = '/kaggle/input/shanghaitech'
PROCESSED_DIR = '/kaggle/working/processed'
CKPT_DIR = '/kaggle/working/checkpoints'
OUT_DIR = '/kaggle/working/outputs'

import os
for d in (PROCESSED_DIR, CKPT_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
print(os.listdir(RAW_DIR))

## 4. Preprocess

In [ ]:
!python -m src.data.preprocess --part B --verify --visualize 3 \
    --set paths.raw_data_dir=$RAW_DIR \
    --set paths.processed_dir=$PROCESSED_DIR \
    --set paths.output_dir=$OUT_DIR \
    --set density.mode=fixed

In [ ]:
!python -m src.data.preprocess --part A --verify --visualize 3 \
    --set paths.raw_data_dir=$RAW_DIR \
    --set paths.processed_dir=$PROCESSED_DIR \
    --set paths.output_dir=$OUT_DIR \
    --set density.mode=adaptive

## 5. Train

Kaggle sessions are capped at 9 GPU hours, so keep an eye on the epoch
budget. `last.pth` is written every epoch; resume from it in a new session.

In [ ]:
PART = 'B'

!python -m src.train \
    --set data.part=$PART \
    --set paths.processed_dir=$PROCESSED_DIR \
    --set paths.checkpoint_dir=$CKPT_DIR \
    --set paths.output_dir=$OUT_DIR \
    --set train.epochs=400 \
    --set train.num_workers=2

### Resume in a later session

`/kaggle/working` is **not** kept between sessions on its own — it only
survives if you explicitly save it. The real steps:

1. **While training is running (or right before your session ends),**
   click **Save Version -> Quick Save** in the top-right menu.
   Quick Save snapshots the current `/kaggle/working` (your
   checkpoints included) as that version's *output*, **without**
   re-running the notebook. (Do **not** use "Save & Run All" here —
   that starts a fresh batch run from cell 1, wiping today's progress
   unless the resume logic below is already wired in from the start.)
2. **In a new session**, re-run the cells above: GPU check, clone,
   paths, **and preprocessing (section 4)** — `$PROCESSED_DIR` is
   also freshly created and empty, and regenerating it only takes
   a minute or two (it's deterministic, so this is safe to redo).
3. Click **Add Input -> Notebook Output Files**, search for *this
   notebook by name*, and attach its latest saved version. It mounts
   read-only at `/kaggle/input/<this-notebook-slug>/`.
4. Set `PREV_OUTPUT` below to that path and run the copy + resume cell.
   Checkpoints are tiny (MCNN has ~134k parameters), so this is fast.

In [ ]:
import shutil

# Edit this to the input mounted in step 3 above, e.g.
# '/kaggle/input/crowd-counting-mcnn-kaggle-training/checkpoints'
PREV_OUTPUT = '/kaggle/input/<this-notebook-slug>/checkpoints'

if os.path.isdir(PREV_OUTPUT):
    shutil.copytree(PREV_OUTPUT, CKPT_DIR, dirs_exist_ok=True)
    print('Copied previous checkpoints into', CKPT_DIR)
    !ls $CKPT_DIR/part_$PART
else:
    raise FileNotFoundError(
        f'{PREV_OUTPUT} not found — attach this notebook\'s previous '
        'saved version as an input first (see markdown above), then '
        'fix PREV_OUTPUT to match its mounted path.'
    )

In [ ]:
# Now that last.pth is sitting in the writable $CKPT_DIR, resume from it.
# New checkpoints/metrics.csv/logs are appended on top of the copied ones.
!python -m src.train \
    --set data.part=$PART \
    --set paths.processed_dir=$PROCESSED_DIR \
    --set paths.checkpoint_dir=$CKPT_DIR \
    --resume $CKPT_DIR/part_$PART/last.pth

## 6. Evaluate

In [ ]:
!python -m src.evaluate \
    --checkpoint $CKPT_DIR/part_$PART/best.pth \
    --part $PART --save-predictions \
    --set paths.processed_dir=$PROCESSED_DIR \
    --set paths.output_dir=$OUT_DIR

In [ ]:
from IPython.display import Image, display
import json

print(json.dumps(json.load(open(f'{OUT_DIR}/eval_part_{PART}.json')), indent=2))
display(Image(f'{OUT_DIR}/scatter_part_{PART}.png'))

## 7. Inference / demo

Kaggle blocks inbound connections, so launch the Gradio app with
`--share` (it tunnels out) or just run the inference CLI on a few images.

In [ ]:
!python -m src.inference \
    --checkpoint $CKPT_DIR/part_$PART/best.pth \
    --input $PROCESSED_DIR/part_$PART/test/images \
    --output $OUT_DIR/predictions

In [ ]:
!python app/gradio_app.py --share --checkpoint-b $CKPT_DIR/part_B/best.pth